# VWAP/Close Rank Alpha Backtest (Yahoo Finance)

Simple cross-sectional alpha:

- Signal per name: `rank(VWAP / Close)`
- Pipeline: market neutralization + gross scaling via `FinStrat.pass_`
- Execution simulator: `FinBT` (daily rebalance)

This notebook uses the default `finTs` historical source (`YFinanceMarketDataProvider`).

In [ ]:
import jax.numpy as jnp
import pandas as pd

from shunya import FinBT, FinStrat, finTs
from shunya.algorithm import cross_section

start_date = "2022-01-01"
end_date = "2025-01-01"
tickers = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL",
    "META", "TSLA", "JPM", "XOM", "UNH",
]

# Keep only columns needed by this alpha so backtest starts from first bar.
panel_columns = ("Open", "High", "Low", "Close", "Volume", "VWAP")

In [ ]:
fts = finTs(
    start_date,
    end_date,
    tickers,
)

def alpha(panel: jnp.ndarray) -> jnp.ndarray:
    # panel columns are exactly panel_columns order above
    close = panel[:, 3]
    vwap = panel[:, 5]
    raw = vwap / close
    return cross_section.rank(raw)

fs = FinStrat(
    fts,
    alpha,
    neutralization="market",
    panel_columns=panel_columns,
)

bt = FinBT(
    fs,
    fts,
    cash=100_000.0,
    commission=0.0005,
    slippage_pct=0.0005,
).run()

results = bt.results(show=False)
pd.Series(results["metrics"]).to_frame("value")

In [ ]:
results["equity_curve"][["Equity", "DrawdownPct"]].tail()